# 05 · MLP (eksploratif, tanpa tuning)
Hyperparameter tetap, 5 seed. Pipeline sama dengan 03 (scaler + PCA di dalam, target distandardisasi). Train saja untuk fit; tidak ada seleksi, jadi Val dan Test hanya dilaporkan.

In [1]:
import sys; sys.path.insert(0, "../src")
import json
import numpy as np, pandas as pd
from sklearn.neural_network import MLPRegressor
import t3
from t3 import CFG, OUT, PRICE, SEED
from eval_utils import *

X = pd.read_parquet(OUT / "features.parquet")
blocks = json.loads((OUT / "blocks.json").read_text())
tr, va, te = (X[X.split == s] for s in ("train", "val", "test"))
SETS = {"A": PRICE, "B": PRICE + blocks["sent_lama"], "fb": PRICE + blocks["finbert"],
        **{t: PRICE + blocks[f"emb_{t}"] for t in CFG["emb_tags"]}}
SEEDS = range(SEED, SEED + 5)

In [2]:
mlp = lambda seed: MLPRegressor(hidden_layer_sizes=(32, 16), alpha=1e-2, max_iter=300, random_state=seed)
rows = []
for s, cols in SETS.items():
    models = [t3.make_pipe(cols, mlp(sd)).fit(tr, tr.target) for sd in SEEDS]
    for name, D in (("Val", va), ("Test", te)):
        ev = pd.DataFrame([evaluate(D.target, m.predict(D)) for m in models])
        rows.append({"set": s, "split": name, **ev.mean()[["MAE", "RMSE", "DirAcc", "PredStd"]], "RMSE_std": ev.RMSE.std(ddof=0),
                     "collapse": is_collapsed(t3.predict(models, D), D.target)})
res = pd.DataFrame(rows)
res.to_csv(OUT / "mlp_results.csv", index=False)
res.round(6)

,set,split,MAE,RMSE,DirAcc,PredStd,RMSE_std,collapse
0,A,Val,0.002729,0.003624,0.513636,0.001678,0.000093,False
1,A,Test,0.002511,0.003230,0.486364,0.001431,0.000084,False
2,B,Val,0.003194,0.004243,0.488636,0.002699,0.000121,False
3,B,Test,0.002900,0.003774,0.505682,0.002421,0.000146,False
4,fb,Val,0.002826,0.003661,0.502273,0.001784,0.000096,False
5,fb,Test,0.002611,0.003337,0.487500,0.001629,0.000028,False
6,bge_small,Val,0.003126,0.004014,0.480682,0.002386,0.000205,False
7,bge_small,Test,0.002751,0.003656,0.517045,0.002105,0.000121,False
8,bge_base,Val,0.003169,0.004117,0.515909,0.002559,0.000157,False
9,bge_base,Test,0.003014,0.003821,0.471591,0.002420,0.000124,False


In [3]:
# Pembanding
for name, D in (("Val", va), ("Test", te)):
    print(name, "| Prediksi 0 RMSE:", round(evaluate(D.target, np.zeros(len(D)))["RMSE"], 6),
          "| Return kemarin RMSE:", round(evaluate(D.target, D.ret)["RMSE"], 6),
          "| DirAcc mayoritas:", round(majority_baselines(D.target)["Mayoritas"], 3))

Val | Prediksi 0 RMSE: 0.003382 | Return kemarin RMSE: 0.004611 | DirAcc mayoritas: 0.534
Test | Prediksi 0 RMSE: 0.002908 | Return kemarin RMSE: 0.003977 | DirAcc mayoritas: 0.591
